<a href="https://colab.research.google.com/github/rheopy/rheolite/blob/main/content/walkthrough/tc_vs_hb_carbopol_pg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥊 TC vs Herschel–Bulkley on a Carbopol flow curve

*Reproduces the [rheofit walkthrough case study](https://rheofit.readthedocs.io/en/latest/walkthrough.html)
in your browser: **2% Carbopol Ultrez 21 in propylene glycol**, a jammed microgel in a viscous
continuous phase — fitted head-to-head with the **TC (three-component)** model and **Herschel–Bulkley**.
Watch TC separate the physics term by term while HB settles for a compromise exponent.*

**Run each cell in order** (▶). The first cell installs `rheofit` in your browser — no server involved.

In [ ]:
%pip install -q rheofit rheopy-rheodata


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rheofit
import rheodata
print("rheofit", getattr(rheofit, "__version__", "installed"), "| rheodata", getattr(rheodata, "__version__", "installed"))


## 🧪 The dataset

A single equilibrium flow sweep of **2% Carbopol Ultrez 21
dispersed in propylene glycol** — 61 points, shear rates 10⁻³–10³ s⁻¹, 20 °C,
concentric-cylinder geometry. The dataset comes from the [rheodata](https://github.com/rheopy/rheodata) library (`caggioni_pg_carbopol_2pct`) — no file download needed.

In [ ]:
df = rheodata.to_rheofit("caggioni_pg_carbopol_2pct", "carbopol_2pct")  # 61 pts


In [ ]:
x = df["Shear rate / 1/s"].to_numpy()
y = df["Stress / Pa"].to_numpy()
T = rheodata.load("caggioni_pg_carbopol_2pct").meta["measurement"]["temperature_C"]
print(f"loaded {len(df)} points, shear rate {x.min():.1e}–{x.max():.1e} 1/s, T = {T:.0f} degC")


## 🥊 The showdown: TC vs Herschel–Bulkley

Carbopol is a jammed microgel — a soft-particle glass — and propylene glycol is a
*viscous* continuous phase. That combination is exactly where the TC (three-component)
model earns its keep over Herschel–Bulkley: HB has no explicit viscous term, so its
exponent drifts into a compromise. Both fits run at `effort="thorough"`, fixed seed —
reproducible bit-for-bit.

In [ ]:
def fit_table(name, label):
    r = rheofit.fit(df, name, effort="thorough", seed=0)
    print(f"--- {label} ---")
    print("RedChi2 = %.3e, cond = %.2g" % (r["redchi"], r["cond"]))
    print("%-14s %12s %12s %10s" % ("param", "value", "stderr", "rel err"))
    for k, v in r["params"].items():
        re = 100 * v["stderr"] / v["value"]
        print("%-14s %12.5g %12.3g %9.2f%%" % (k, v["value"], v["stderr"], re))
    print()
    return r

r_tc = fit_table("tc", "TC")
r_hb = fit_table("herschel_bulkley", "Herschel-Bulkley")
print("TC wins by %.1fx in RedChi2" % (r_hb["redchi"] / r_tc["redchi"]))

p = {m: {k: v["value"] for k, v in r["params"].items()}
     for m, r in [("tc", r_tc), ("hb", r_hb)]}
g = np.logspace(np.log10(x.min()), np.log10(x.max()), 400)
s_tc = p["tc"]["sigma_y"] + p["tc"]["sigma_y"]*np.sqrt(g/p["tc"]["gamma_dot_c"]) + p["tc"]["eta_bg"]*g
s_hb = p["hb"]["sigma_y"] + p["hb"]["K"]*g**p["hb"]["n"]

## 📉 What the numbers say

| model | RedChi² | parameters |
|---|---|---|
| **TC** | **6.018e-04** | σ_y = 20.8 Pa, γ̇_c = 0.95 s⁻¹, η_bg = 0.715 Pa·s |
| Herschel–Bulkley | 3.651e-03 | σ_y = 22.0 Pa, K = 19.2 Pa·sⁿ, n = 0.595 |

TC wins by a factor of ~6 in relative-weighted error, and every TC
parameter is tightly identified (relative errors 0.6–3.0%,
no warnings). But the number is the least interesting part — look at *where* HB fails,
in the residual panel of the plot below.

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
a1.loglog(x, y, "ko", ms=4, label="data (Carbopol 2% in PG)")
a1.loglog(g, s_tc, "r-", lw=2, label="TC (RedChi2=%.2e)" % r_tc["redchi"])
a1.loglog(g, s_hb, "b--", lw=1.8, label="Herschel–Bulkley (RedChi2=%.2e)" % r_hb["redchi"])
a1.set_ylabel("stress / Pa")
a1.set_title("TC vs Herschel–Bulkley on the Carbopol flow curve")
a1.legend(); a1.grid(True, which="both", alpha=0.3)

res_tc = (y - (p["tc"]["sigma_y"] + p["tc"]["sigma_y"]*np.sqrt(x/p["tc"]["gamma_dot_c"]) + p["tc"]["eta_bg"]*x)) / y * 100
res_hb = (y - (p["hb"]["sigma_y"] + p["hb"]["K"]*x**p["hb"]["n"])) / y * 100
a2.semilogx(x, res_tc, "ro", ms=3, label="TC")
a2.semilogx(x, res_hb, "bs", ms=3, label="HB")
a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel("shear rate / 1/s"); a2.set_ylabel("rel. resid. / %")
a2.legend(fontsize=8); a2.grid(True, which="both", alpha=0.3)


HB's residuals swing in a systematic S-shape (±10–20%) while TC's stay flat (±5%).
A single power-law exponent *n* ≈ 0.6 is trying to do two jobs at once — mimic the √γ̇
plastic rise at intermediate rates *and* the linear viscous rise at high rates — and it
does neither exactly. This is the physics the walkthrough argues: the HB exponent is an
empirical compromise that drifts with the continuous-phase viscosity, while TC gives
each dissipation mechanism its own parameter.

## 🔬 Reading the TC fit: three mechanisms, one curve

$$\sigma = \underbrace{\sigma_y}_{\text{elastic}} +
\underbrace{\sigma_y\sqrt{\dot\gamma/\dot\gamma_c}}_{\text{plastic}} +
\underbrace{\eta_{bg}\,\dot\gamma}_{\text{viscous}}$$

Fitted to the data, each term draws its own curve — and each curve means something.

In [ ]:
sy, gdc, eb = p["tc"]["sigma_y"], p["tc"]["gamma_dot_c"], p["tc"]["eta_bg"]
t_el  = np.full_like(g, sy)                    # elastic: rate-independent plateau
t_pl  = sy * np.sqrt(g / gdc)                 # plastic: elastoplastic rearrangements
t_vi  = eb * g                                # viscous: continuous-phase dissipation

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(x, y, "ko", ms=4, label="data")
ax.loglog(g, s_tc, "k-", lw=2, label="TC fit (sum)")
ax.loglog(g, t_el, "r--", lw=1.8, label="elastic  σ_y = %.1f Pa" % sy)
ax.loglog(g, t_pl, "tab:orange", ls="--", lw=1.8, label="plastic  γ̇_c = %.2f s⁻¹" % gdc)
ax.loglog(g, t_vi, "b--", lw=1.8, label="viscous  η_bg = %.3f Pa·s" % eb)
ax.set_xlabel("shear rate / 1/s"); ax.set_ylabel("stress / Pa")
ax.set_title("TC three-term decomposition of the Carbopol flow curve")
ax.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)


Which mechanism carries the stress depends on where you look — the *dissipation map*
below shows each term's fraction of the total stress.

In [ ]:
f_el = t_el / s_tc
f_pl = t_pl / s_tc
f_vi = t_vi / s_tc

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(g, f_el, "r-", lw=2, label="elastic (σ_y)")
ax.semilogx(g, f_pl, color="tab:orange", lw=2, label="plastic (√γ̇)")
ax.semilogx(g, f_vi, "b-", lw=2, label="viscous (η_bg γ̇)")
ax.axhline(1.0, color="k", lw=0.6, alpha=0.4)
ax.set_xlabel("shear rate / 1/s"); ax.set_ylabel("fraction of total stress")
ax.set_title("Dissipation map: who carries the stress?")
ax.set_ylim(-0.05, 1.05)
ax.legend(); ax.grid(True, which="both", alpha=0.3)


## 💡 Takeaway

- 🔴 **Elastic, σ_y = 20.8 Pa** — the rate-independent plateau. The jammed microgel network holds until stress exceeds this; below it, the sample is a solid.
- 🟠 **Plastic, γ̇_c = 0.95 s⁻¹** — the √γ̇ rise from elastoplastic rearrangements (Hébraud–Lequeux / kinetic elastoplastic picture). γ̇_c sets *where* this regime lives.
- 🔵 **Viscous, η_bg = 0.715 Pa·s** — the linear high-shear tail: dissipation through the propylene glycol continuous phase. In water this term would be negligible; in propylene glycol it bends the whole top end of the curve — which is exactly why HB's single exponent fails here.

At rest and low rates the response is almost entirely elastic (the yield stress you feel
when the gel won't pour). Through the middle decades, plastic rearrangements take over.
At the highest rates, the viscous continuous phase dominates. Six decades, three
regimes, three parameters — each one tied to something you could formulate *against*:
raise σ_y with more Carbopol, shift γ̇_c with the microgel softness, tune η_bg with the
solvent.

*Full write-up: the [rheofit case study](https://rheofit.readthedocs.io/en/latest/walkthrough.html)
(Caggioni, Trappe & Spicer, J. Rheol. **64**, 413–422 (2020), doi:10.1122/1.5120633).*